In [14]:
# stack 구현하기 문제풀이 (후위 표기법)

# 직접 구현한 스택을 이용한 함수
# 중위 표기식(expression)을 받아 후위 표기식 문자열을 반환
def to_postfix(expression: str) -> str:
  op: dict[str, int] = {"+": 1, "-": 1, "*": 2, "/": 2}
# 연산자 우선순위와 상관없음. value 로만 우선순위를 판단한다.
  res : str = ""
  s = Stack()


  for exp in expression:  # 문자를 하나씩 확인
    if exp.isnumeric(): # 숫자라면
      res += exp  # res (결과)에 exp를 추가한다

    elif exp in op:
      # stack top 의 우선순위가 같거나 높으면 pop
      if not s.is_empty() and (op[exp]<= op[s.peek()]):
        res += s.pop()
      s.push(exp) # 트러블 슈팅 : if 문 밖에서 push 할 것
      # 현재 연산자는 stack 에 push
  while not s.is_empty(): # 끝까지 처리안된 연산자 꺼내기
    res += s.pop()  # 남은연산자 전부 pop
  return res


# 테스트 코드
for expr in ("3 + 5 * 2", "3 * 5 + 2"):
  print(f"{expr} -> {to_postfix(expr)}")

3 + 5 * 2 -> 352*+
3 * 5 + 2 -> 35*2+


In [17]:
# 후위표기법2 (괄호까지 처리하는 프로그램)
# 직접 구현한  Stack 클래스 사용
def to_postfix2(expression: str) -> str:
    op: dict[str, int] = {"+": 1, "-": 1, "*": 2, "/": 2}
    res: str = ""
    s: list[str] = []
# 파이썬 내장 리스트 사용 .append(), .pop(), s[-1],if s
    for exp in expression:
        if exp.isnumeric(): # 숫자는
            res += exp  # 결과에 추가

        elif exp == "(":
            s.append(exp) # 스택에 넣고 대기

        elif exp == ")":
            while s[-1] != "(": # "(" 만날 때 까지
                res += s.pop()  # 연산자들 pop
            s.pop()  # 불필요한 "("제거

        elif exp in op:
            if s and s[-1] != "(" and (op[exp] <= op[s[-1]]):
              # 조건 : 스택있고 & top이 "(" 이 아니고, &. 우선순위 비교
                res += s.pop()
            s.append(exp)
    # 남은 연산자 처리
    while s:
        res += s.pop()

    return res


# 테스트 코드
for expr in ("(3 + 5) * 2", "((1 + 2) * 3) / 4 + 5 * (6 - 7)"):
    print(f"{expr} -> {to_postfix2(expr)}")



(3 + 5) * 2 -> 35+2*
((1 + 2) * 3) / 4 + 5 * (6 - 7) -> 12+3*4/567-*+


In [22]:
# 후위표기법 계산기(완전체)
def eval_postfix(expression: str) -> int:
    s: list[int] = []  # 숫자 저장용 스택 (파이썬 리스트 사용)

    for exp in expression:
        if exp.isnumeric():  # 숫자면
            s.append(int(exp))  # 정수로 변환해서 스택에 push

        elif exp != " ":  # 연산자면 (공백 무시)
            n2 = s.pop()  # 두 번째 피연산자 (나중에 들어간 것)
            n1 = s.pop()  # 첫 번째 피연산자 (먼저 들어간 것)
            # 순서 중요: n1 (연산자) n2 로 계산

            if exp == "+":
                res = n1 + n2
            elif exp == "-":
                res = n1 - n2
            elif exp == "*":
                res = n1 * n2
            else:  # "/"
                res = n1 / n2

            s.append(res)  # 계산 결과를 다시 스택에 push

    return s[0]  # 최종 결과 (스택에 남은 마지막 값)


# 테스트 코드
for expr in ("35+2*", "12+3*4/567-*+"):
    print(f"{expr} -> {eval_postfix(expr)}")

35+2 -> 8
12+3*4/567-*+ -> -2.75


In [23]:
'''
자신보다 큰 원소 찾기
음이 아닌 정수 배열이 주어졌을 때, 각 원소의 오른쪽에 있는 원소 중에서
현재 원소보다 큰 값을 출력하되, 가장 근접한 원소를 출력할아. 현재 원소보다
큰 값이 없으면 -1을 출력하라 .
'''

# 이중for 문 을 이용한 방법 (직관적이지만 느리다. 코테등을 생각한다면 좋지 않은 방법. 그냥 이런 방법도 있다..)
def find_nge(arr : list[int]) -> list[int]:
  n : int = len(arr)
  for i in range(n):
    nge: int = -1
    for j in range(i+1, n):
      if arr[j] > arr[i]:
        nge = arr[j]
        break
    print(f"{arr[i]} --> {nge}")


# 테스트 코드
find_nge([4, 5, 2, 25])


4 --> 5
5 --> 25
2 --> 25
25 --> -1


In [24]:
# 스택을 이용한 방법 (간결,명확)
def find_nge(arr :  list[int]) -> list[int]:
  n: int = len(arr)
  s: list[int] = [] # 스택 (오른쪽 큰 값 후보들)
  res : list [int] = [-1] * n # 결과배열 ,-1로 초기화

  for i in range(n-1, -1, -1):  # 오른쪽에서 왼쪽으로
    while s:  # 스택에 값이 있는동안
      if s[-1] > arr[i]:  # 나보다 큰 값 찾음
        res[i] = s[-1]
        break
      else: # 나보다 작다면
        s.pop() # 필요없으니 버림
    s.append(arr[i])  # 현재 값 스택에 추가
  for i in range(n):
    print(f"{arr[i]} --> {res[i]}")

# 테스트 코드
find_nge([4, 5, 2, 25])

4 --> 5
5 --> 25
2 --> 25
25 --> -1
